# Pilot100 ile 5 Encoder — Pilot600 Protokolü

Bu notebook `train_morph_encoder_pilot600_colab.ipynb` ile **aynı eğitim protokolünü** beş encoder için sırayla uygular:

1. Pilot100'ün tamamı LoRA eğitimine girer.
2. Sealed 600, eğitimden önce sabit ve feature-dengeli 100 validation + 500 final test olarak ayrılır.
3. Her model 16 epoch tamamlanır; validation MRR@10'u en yüksek LoRA geri yüklenir, controlled loss tanı metriği olarak izlenir.
4. Her fine-tuned model aynı 500-family final testte ölçülür.

Sonuçlar yalnız temel validation ve final-test metrikleriyle tek tabloda karşılaştırılır.


## 1. Colab kurulumu


In [ ]:
# Colab'ın önceden kurulu torchvision paketi bazı torch sürümleriyle NMS çakışması yapıyor.
# Bu text-only eğitimde torchvision ve eski torchao gerekmediği için kaldırıyoruz.
!pip uninstall -y -q torchvision torchao
!pip install -q -U "sentence-transformers>=3.1.0" "transformers>=4.48.0" peft accelerate datasets pandas

import gc, hashlib, json, os, random, shutil, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch

print("torch", torch.__version__)
print("GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK")
assert torch.cuda.is_available(), "GPU runtime seçin."


## Ayarlar

In [ ]:
REPO_URL = "https://github.com/TR-morph-retrieval/turkish-morph-retrieval.git"
ROOT = Path("/content/turkish-morph-retrieval")
if not (ROOT / ".git").exists():
    if ROOT.exists(): shutil.rmtree(ROOT)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

INSTRUCT = "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"
MODELS = {
    "modernbert-tr": {"repo":"ytu-ce-cosmos/modernbert-tr-embed", "q":INSTRUCT, "d":"", "targets":["Wqkv","Wo","Wi"]},
    "bge-m3": {"repo":"BAAI/bge-m3", "q":"", "d":"", "targets":["query","key","value","dense"]},
    "e5-large": {"repo":"intfloat/multilingual-e5-large", "q":"query: ", "d":"passage: ", "targets":["query","key","value","dense"]},
    "trmteb-ft-110m": {"repo":"trmteb/turkish-embedding-model-fine-tuned", "q":"", "d":"", "targets":["query","key","value","dense"]},
    "multilingual-mpnet": {"repo":"sentence-transformers/paraphrase-multilingual-mpnet-base-v2", "q":"", "d":"", "targets":["query","key","value","dense"]},
}
SEED = 42
PILOT_MODE = True
TRAIN_JSONL = ROOT / "train/archive/pilot/data/pilot100.jsonl"
PILOT_VALIDATION_SIZE = 100
RUN_SEALED_TEST = True
# Pilot600 notebook ile aynı değerler.
PARAMS = dict(epochs=16, batch_size=32, cache_mini_batch=16, learning_rate=1.5e-5,
              warmup_ratio=.1, weight_decay=.01, lora_r=16, lora_alpha=32,
              lora_dropout=.05, max_seq_length=256)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("models:", list(MODELS), "params:", PARAMS)


## Pilot100 ve Pilot600 evaluation split

In [ ]:
from datasets import Dataset

def stable_dev(family_id, seed=SEED):
    h = hashlib.sha256(f"{seed}:{family_id}".encode()).digest()
    return int.from_bytes(h[:4], "big") % 10 == 0

def load_rows(path, pilot_mode):
    if not path.exists():
        raise FileNotFoundError(f"Train JSONL bulunamadı: {path}")
    raw = [json.loads(x) for x in path.read_text(encoding="utf-8").splitlines() if x.strip()]
    if not raw: raise ValueError("Train JSONL boş")
    if pilot_mode:
        from train.production import validate_family
        kept, skipped = [], []
        for record in raw:
            errors = validate_family(record["family"])
            (skipped if errors else kept).append((record, errors))
        raw = [record for record, _ in kept]
        print(f"Pilot filtresi: {len(raw)} kullanılacak, {len(skipped)} eski/uyumsuz kayıt atlandı.")
    else:
        bad = [r.get("slot_id") for r in raw
               if r.get("purpose") == "pilot_only" or r.get("eligible_for_final_train") is not True]
        if bad: raise ValueError(f"Pilot/onaysız kayıt final train'e karışmış: {bad[:5]}")
    rows=[]
    for r in raw:
        f=r["family"]; by={c["slot"]:c["text"].strip() for c in f["candidates"]}
        required={"positive","morph_1","morph_2","semantic_1"}
        if set(by) != required: raise ValueError(f"Aday şeması hatalı: {f['family_id']}")
        # Eski pilot kayıtlarında family_id yerine benzersiz pilot_review_id tutulmuş olabilir.
        family_id=f.get("family_id") or r.get("pilot_review_id")
        if not family_id: raise ValueError(f"Kimlik eksik: {r.get('slot_id', '<bilinmiyor>')}")
        rows.append(dict(family_id=family_id, target_feature=f["target_feature"],
                         query=f["query"].strip(), positive=by["positive"],
                         morph_1=by["morph_1"], morph_2=by["morph_2"],
                         semantic_1=by["semantic_1"]))
    ids=[r["family_id"] for r in rows]
    if len(ids)!=len(set(ids)): raise ValueError("Tekrarlı family_id")
    return rows

ALL_ROWS=load_rows(TRAIN_JSONL, PILOT_MODE)
# Pilotta train içi validation yok: 60 family'nin tamamı LoRA eğitimine girer.
TRAIN_ROWS=ALL_ROWS if PILOT_MODE else [r for r in ALL_ROWS if not stable_dev(r["family_id"])]
DEV_ROWS=[] if PILOT_MODE else [r for r in ALL_ROWS if stable_dev(r["family_id"])]
assert TRAIN_ROWS
print(f"{'pilot' if PILOT_MODE else 'final'}={len(ALL_ROWS)} train={len(TRAIN_ROWS)} dev={len(DEV_ROWS)}")
display(pd.Series([r['target_feature'] for r in ALL_ROWS]).value_counts().describe().to_frame('fenomen dağılımı'))

# 600 değerlendirme family'sini sabit 100 development + 500 final test olarak ayır.
# Bu seçim eğitimden önce yapılır ve her run'da aynı family'leri verir.
from collections import defaultdict
shard_dir=ROOT/"test/data/final_shards"
all_eval_items=[json.loads(line) for sf in sorted(shard_dir.glob("*.jsonl"))
                for line in sf.read_text(encoding="utf-8").splitlines() if line.strip()]
assert len(all_eval_items)==600 and all(x.get("target_split")=="sealed_test" for x in all_eval_items)
by_feature=defaultdict(list)
for item in all_eval_items: by_feature[item["target_feature"]].append(item)
rank_key=lambda x: hashlib.sha256(f"{SEED}:{x['family_id']}".encode()).hexdigest()
PILOT_VAL_ITEMS=[sorted(group,key=rank_key)[0] for _,group in sorted(by_feature.items())]
remaining=[x for x in all_eval_items if x not in PILOT_VAL_ITEMS]
PILOT_VAL_ITEMS += sorted(remaining,key=rank_key)[:PILOT_VALIDATION_SIZE-len(PILOT_VAL_ITEMS)]
PILOT_VAL_IDS={x["family_id"] for x in PILOT_VAL_ITEMS}
FINAL_TEST_ITEMS=[x for x in all_eval_items if x["family_id"] not in PILOT_VAL_IDS]
assert len(PILOT_VAL_ITEMS)==PILOT_VALIDATION_SIZE and len(FINAL_TEST_ITEMS)==600-PILOT_VALIDATION_SIZE
print(f"evaluation split: dev={len(PILOT_VAL_ITEMS)} (feature-dengeli), final_test={len(FINAL_TEST_ITEMS)}")


## Beş encoder: Pilot600 eğitim ve değerlendirme döngüsü

In [ ]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.losses import CachedMultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.evaluation import SentenceEvaluator
from peft import LoraConfig, get_peft_model
from transformers import TrainerCallback
from test.evaluation import score_encoder

def train_dataset_for(cfg):
    qtext=lambda x: cfg["q"] + x
    dtext=lambda x: cfg["d"] + x
    samples=[]
    for r in TRAIN_ROWS:
        samples.append({"anchor":qtext(r["query"]), "positive":dtext(r["positive"]),
                        "negative_1":dtext(r["morph_1"]), "negative_2":dtext(r["morph_2"]),
                        "negative_3":dtext(r["semantic_1"])})
        samples.append({"anchor":qtext(r["positive"]), "positive":dtext(r["query"]),
                        "negative_1":dtext(r["morph_1"]), "negative_2":dtext(r["morph_2"]),
                        "negative_3":dtext(r["semantic_1"])})
    return Dataset.from_list(samples)

class ControlledRecallEvaluator(SentenceEvaluator):
    def __init__(self, items, cfg, name="pilot_validation"):
        self.items, self.cfg, self.name = items, cfg, name
        # Checkpoint seçimi doğrudan retrieval sıralamasıyla uyumlu olsun.
        self.primary_metric=f"{name}_mrr_at_10"
        self.greater_is_better=True
    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        result=score_encoder(model,self.items,query_prefix=self.cfg["q"],document_prefix=self.cfg["d"],batch_size=32)
        summary, score_map=result["summary"], result["scores"]
        losses=[]
        for item in self.items:
            values=np.asarray([score_map[item["family_id"]][c["id"]] for c in item["candidates"]],dtype=float)*20.0
            gold_index=next(i for i,c in enumerate(item["candidates"]) if c["id"]==item["gold_id"])
            shifted=values-values.max()
            losses.append(float(-shifted[gold_index]+np.log(np.exp(shifted).sum())))
        return {f"{self.name}_controlled_loss":float(np.mean(losses)),
                f"{self.name}_recall_at_1":summary["recall@1"],
                f"{self.name}_recall_at_3":summary["recall@3"],
                f"{self.name}_mrr_at_10":summary["mrr@10"]}

class RestoreBestLoRA(TrainerCallback):
    def __init__(self, metric_name, greater_is_better):
        self.metric_name=metric_name; self.greater_is_better=greater_is_better
        self.best_score=float("-inf") if greater_is_better else float("inf")
        self.best_state=None; self.best_epoch=None
    def on_evaluate(self,args,state,control,metrics=None,model=None,**kwargs):
        score=(metrics or {}).get(f"eval_{self.metric_name}", (metrics or {}).get(self.metric_name))
        improved=score is not None and (score > self.best_score if self.greater_is_better else score < self.best_score)
        if improved:
            self.best_score=float(score); self.best_epoch=state.epoch
            self.best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items() if "lora_" in k}
        return control

def run_one(key,cfg):
    print(f"\n===== {key} =====")
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model=SentenceTransformer(cfg["repo"],trust_remote_code=True,model_kwargs={"torch_dtype":dtype})
    model.max_seq_length=PARAMS["max_seq_length"]
    base_val=score_encoder(model,PILOT_VAL_ITEMS,query_prefix=cfg["q"],document_prefix=cfg["d"],batch_size=32)["summary"]
    base_test=score_encoder(model,FINAL_TEST_ITEMS,query_prefix=cfg["q"],document_prefix=cfg["d"],batch_size=32)["summary"]
    evaluator=ControlledRecallEvaluator(PILOT_VAL_ITEMS,cfg)
    first=model._first_module()
    available={name.rsplit(".",1)[-1] for name,_ in first.auto_model.named_modules()}
    targets=[name for name in cfg["targets"] if name in available]
    if not targets: raise RuntimeError(f"LoRA hedefi bulunamadı: {key}; {sorted(available)[:40]}")
    first.auto_model=get_peft_model(first.auto_model,LoraConfig(r=PARAMS["lora_r"],lora_alpha=PARAMS["lora_alpha"],lora_dropout=PARAMS["lora_dropout"],bias="none",target_modules=targets,task_type="FEATURE_EXTRACTION"))
    # Eğitim öncesi LoRA adapter'ı teorik olarak sıfır etkili olmalı; aksi halde sonuçları yorumlamayız.
    peft_init_val=score_encoder(model,PILOT_VAL_ITEMS,query_prefix=cfg["q"],document_prefix=cfg["d"],batch_size=32)["summary"]
    sanity_keys=("recall@1","recall@3","mrr@10","ndcg@10")
    peft_init_delta={metric:peft_init_val[metric]-base_val[metric] for metric in sanity_keys}
    peft_init_ok=all(abs(delta)<1e-12 for delta in peft_init_delta.values())
    print("LoRA başlangıç kontrolü:", "OK" if peft_init_ok else f"UYARI {peft_init_delta}")
    loss=CachedMultipleNegativesRankingLoss(model,scale=20.0,mini_batch_size=PARAMS["cache_mini_batch"])
    best_lora=RestoreBestLoRA(evaluator.primary_metric,evaluator.greater_is_better)
    args=SentenceTransformerTrainingArguments(output_dir=f"/content/pilot600_compare_{key}",num_train_epochs=PARAMS["epochs"],per_device_train_batch_size=PARAMS["batch_size"],learning_rate=PARAMS["learning_rate"],warmup_ratio=PARAMS["warmup_ratio"],weight_decay=PARAMS["weight_decay"],fp16=not torch.cuda.is_bf16_supported(),bf16=torch.cuda.is_bf16_supported(),batch_sampler=BatchSamplers.NO_DUPLICATES,logging_strategy="steps",logging_steps=1,logging_first_step=True,eval_strategy="epoch",save_strategy="no",load_best_model_at_end=False,eval_on_start=True,seed=SEED,data_seed=SEED,report_to="none")
    trainer=SentenceTransformerTrainer(model=model,args=args,train_dataset=train_dataset_for(cfg),loss=loss,evaluator=evaluator,callbacks=[best_lora])
    trainer.train()
    if best_lora.best_state: trainer.model.load_state_dict(best_lora.best_state,strict=False)
    model=trainer.model
    tuned_val=score_encoder(model,PILOT_VAL_ITEMS,query_prefix=cfg["q"],document_prefix=cfg["d"],batch_size=32)["summary"]
    tuned_test=score_encoder(model,FINAL_TEST_ITEMS,query_prefix=cfg["q"],document_prefix=cfg["d"],batch_size=32)["summary"] if RUN_SEALED_TEST else None
    history=pd.DataFrame(trainer.state.log_history)
    result={"model":key,"repo":cfg["repo"],"targets":targets,"best_epoch":best_lora.best_epoch,"selection_metric":evaluator.primary_metric,"best_validation_score":best_lora.best_score,"peft_init_validation_matches_base":peft_init_ok,"peft_init_validation_delta":peft_init_delta,"zero_validation":base_val,"fine_tuned_validation":tuned_val,"zero_test":base_test,"fine_tuned_test":tuned_test,"training_log":history.to_dict("records")}
    del trainer,loss,model; gc.collect(); torch.cuda.empty_cache()
    return result

results=[]
for key,cfg in MODELS.items():
    try: results.append(run_one(key,cfg))
    except Exception as exc:
        print("FAILED",key,repr(exc)); results.append({"model":key,"error":repr(exc)})

Path("/content/pilot600_compare5_results.json").write_text(json.dumps(results,ensure_ascii=False,indent=2,default=str),encoding="utf-8")


## 500-family sealed test — sonuç özeti

Bu hücreyi beş modelin eğitim hücresinden sonra çalıştırın. Eğitim boyunca dokunulmayan 500 family için zero-shot ve en iyi LoRA sonuçlarını karşılaştırır.


In [ ]:
rows=[]
for result in results:
    if "error" in result:
        rows.append({"model":result["model"],"error":result["error"]}); continue
    for stage,metrics in [("zero-shot / validation_100",result["zero_validation"]),("best LoRA / validation_100",result["fine_tuned_validation"]),("zero-shot / final_test_500",result["zero_test"]),("best LoRA / final_test_500",result["fine_tuned_test"])]:
        rows.append({"model":result["model"],"stage":stage,"best_epoch":result["best_epoch"],"selection_metric":result["selection_metric"],"best_validation_score":result["best_validation_score"],**metrics})
comparison=pd.DataFrame(rows)
# Ayrıntılı 20 satır dosyaya yazılır; çıktı alanında yalnız karar için gerekli özet kalır.
comparison.to_csv("/content/pilot600_compare5_metrics.csv",index=False)
basic=["model","stage","recall@1","recall@3","mrr@10","ndcg@10","pairwise_morph_hard_accuracy","pairwise_semantic_hard_accuracy","contrast_consistency"]
sealed=comparison[comparison.get("stage",pd.Series(dtype=str)).astype(str).str.contains("final_test_500",na=False)]
print("500-family sealed test (yalnız final sonuçlar):")
display(sealed[[c for c in basic if c in sealed.columns]])
dev_best=comparison[comparison.get("stage",pd.Series(dtype=str)).astype(str).eq("best LoRA / validation_100")]
print("Validation ile seçilen en iyi epoch (yalnız seçme bilgisi):")
display(dev_best[[c for c in ["model","best_epoch","selection_metric","best_validation_score","recall@1","mrr@10"] if c in dev_best.columns]])
print("Ayrıntılı 20-satırlı tablo: /content/pilot600_compare5_metrics.csv")
assert not sealed.empty and sealed["model"].nunique()==len(MODELS), "Bir model hata verdi; eğitim hücresindeki FAILED satırını kontrol edin."
